# Deploy NPS Agent to RHOAI

This notebook deploys the NPS Agent ([`npsagent.py`](./npsagent.py)) to OpenShift AI with MLflow tracing.

## Overview

This notebook takes the agent from the [Evaluate notebook](../1_develop/2_evaluate.ipynb) and deploys it as an HTTP endpoint on **Red Hat OpenShift AI (RHOAI)**. The agent logic is unchanged — we just wrap it in an MLflow `ResponsesAgent` for serving.


### Cluster Setup

Before continuing, you'll need an OpenShift cluster with RHOAI and MLflow already configured. Follow the [Cluster Setup Guide](https://docs.google.com/document/d/1ZzuGAY1gSamOLsznwbaL7xFkJ1JjHWpnyjk0tV12YVg/edit?tab=t.0#heading=h.jgt5ddlrwyvc) to get your environment ready.

---

## Deployment Files Overview

The `2_deploy/` directory is self-contained — everything OpenShift needs to build and run the agent:

| File | What it does |
|---|---|
| [`npsagent.py`](./npsagent.py) | The agent logic (identical to the evaluate notebook) wrapped in an MLflow `ResponsesAgent` so it can serve HTTP requests via `POST /invocations` |
| [`nps_mcp_server.py`](./nps_mcp_server.py) | FastMCP server exposing NPS API tools. Spawned on-demand by `run_nps_agent` via `uv run fastmcp run` — not a long-running process |
| [`app.sh`](./app.sh) | Container entry point. Packages `npsagent.py` with `mlflow.pyfunc.save_model`, then starts `mlflow models serve` on port 8080 (5 min timeout) |
| [`requirements.txt`](./requirements.txt) | Python dependencies installed during the s2i build |
| [`nps-agent.yaml`](./nps-agent.yaml) | OpenShift Template that creates a **BuildConfig** (s2i from `deploydemo` branch using `python:3.12-ubi9`), **ImageStream**, **Deployment** (with secret injection + readiness/liveness probes on `/ping`), **Service** (port 8080), and **Route** (HTTPS with 5 min HAProxy timeout) — all in one `oc process` call |
| [`.s2i/environment`](./.s2i/environment) | Single line: `APP_SCRIPT=app.sh` — tells the [s2i](https://github.com/openshift/source-to-image) builder to use `app.sh` instead of the default Python entrypoint |

---

## What's Different in the Deployed Version?

The core `run_nps_agent` in [`npsagent.py`](./npsagent.py) is identical to the evaluate notebook. The only additions are described below.

### `NPSResponsesAgent` — MLflow Serving Wrapper

MLflow serves models over HTTP via a `POST /invocations` endpoint. To plug our agent into this, we subclass [`ResponsesAgent`](https://mlflow.org/docs/latest/python_api/mlflow.pyfunc.html) — MLflow's standard interface for chat-style models. The `predict` method receives the request, extracts the user message, calls `run_nps_agent`, and returns the result in MLflow's Responses format. This is what turns our agent into a deployable HTTP service.

### RHOAI Workspace Header

RHOAI's Data Science Gateway uses an `X-Mlflow-Workspace` header to route traces to the correct project. When `MLFLOW_WORKSPACE` is set (i.e. running on RHOAI), `npsagent.py` registers a custom `RequestHeaderProvider` that attaches this header to every MLflow tracking request. Locally this is a no-op.

See [`npsagent.py`](./npsagent.py) for the full code.

---

## Prerequisites

Before you begin, make sure you have:

- An OpenShift cluster with RHOAI and MLflow configured
- The `oc` CLI installed and logged in to your cluster
- An [OpenAI API key](https://platform.openai.com/api-keys)
- An [NPS API key](https://www.nps.gov/subjects/developer/get-started.htm) (free and instant)
- A `.env` file in the repository root with your `OPENAI_API_KEY` and `NPS_API_KEY`

In [ ]:
import os
import subprocess
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env")

## OpenAI Environment Variables
os.environ.setdefault("OPENAI_API_KEY", "")
os.environ.setdefault("OPENAI_BASE_URL", "https://api.openai.com/v1")
os.environ.setdefault("OPENAI_MODEL_NAME", "gpt-4o-mini")

## NPS API Key
os.environ.setdefault("NPS_API_KEY", "")

# Check that required vars are set
required_vars = ["OPENAI_API_KEY", "OPENAI_BASE_URL", "OPENAI_MODEL_NAME", "NPS_API_KEY"]
if any(not os.getenv(var) for var in required_vars):
    raise ValueError("One or more required environment variables are not set. Check your .env file.")

## Step 1 — Create an OpenShift Project

Each deployment lives in its own OpenShift namespace. Set yours below — use `nps-agent-<yourname>` to avoid conflicts with other users on the same cluster. `MLFLOW_TRACKING_URI` is read from your `.env` file.

In [ ]:
NAMESPACE = ""

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "")
MLFLOW_WORKSPACE = NAMESPACE
MLFLOW_EXPERIMENT_NAME = "nps-agent"

In [ ]:
!oc new-project {NAMESPACE}

## Step 2 — Create the Secret with API Keys

Store your API keys as an OpenShift Secret so the running pod can access them without baking credentials into the image.

In [ ]:
!oc create secret generic nps-agent-secrets \
  --from-literal=OPENAI_API_KEY="{os.getenv('OPENAI_API_KEY')}" \
  --from-literal=NPS_API_KEY="{os.getenv('NPS_API_KEY')}" \
  -n {NAMESPACE}

## Step 3 — Apply the OpenShift Template

Process [`nps-agent.yaml`](./nps-agent.yaml) with your cluster values and apply all resources (see [Deployment Files](#Deployment-Files-Overview) above for what each resource does).

In [ ]:
!oc process -f ./nps-agent.yaml \
  -p NAMESPACE="{NAMESPACE}" \
  -p MLFLOW_TRACKING_URI="{MLFLOW_TRACKING_URI}" \
  -p MLFLOW_WORKSPACE="{MLFLOW_WORKSPACE}" \
  -p MLFLOW_EXPERIMENT_NAME="{MLFLOW_EXPERIMENT_NAME}" \
  | oc apply -f -

## Step 4 — Wait for the s2i Build

The BuildConfig triggers automatically. Watch until you see **"Push successful"**.

In [ ]:
!oc logs -f build/nps-agent-1 -n {NAMESPACE}

## Step 5 — Set the MLflow Auth Token

The RHOAI Data Science Gateway requires an auth token. Set it from your current `oc` session.

> **Note:** `oc` tokens expire. Re-run this cell when you need to refresh.

In [ ]:
!oc set env deployment/nps-agent \
  MLFLOW_TRACKING_TOKEN="$(oc whoami -t)" \
  -n {NAMESPACE}

## Step 6 — Verify the Pod is Running

Once the build completes and the image is pushed, the Deployment rolls out a pod. Check that it's in `Running` state before continuing.

In [177]:
!oc get pods -n {NAMESPACE}

Error from server (Forbidden): pods is forbidden: User "nnarendr@redhat.com" cannot list resource "pods" in API group "" in the namespace "nehanth"


## Step 7 — Get the Route URL

The OpenShift Route exposes the agent pod as a public HTTPS endpoint. Grab the hostname so we can call it.

In [ ]:
ROUTE_HOST = subprocess.check_output(
    ["oc", "get", "route", "nps-agent", "-n", NAMESPACE, "-o", "jsonpath={.spec.host}"]
).decode().strip()

AGENT_URL = f"https://{ROUTE_HOST}"
print(f"Agent URL: {AGENT_URL}")

## Step 8 — Test the Agent

Send a sample question to the deployed agent's `/invocations` endpoint and display the response.

In [ ]:
import requests
from IPython.display import display, Markdown

payload = {
    "input": [
        {"role": "user", "content": "What national parks are in California?"}
    ]
}

resp = requests.post(
    f"{AGENT_URL}/invocations",
    headers={"Content-Type": "application/json"},
    json=payload,
    timeout=300,
)

print(f"Status: {resp.status_code}")
if resp.status_code == 200:
    result = resp.json()
    output_text = result.get("output", [{}])[0].get("text", str(result))
    display(Markdown(output_text))
else:
    print(f"Error: {resp.text}")

## Step 9 — View Traces in MLflow

Open your RHOAI MLflow UI and navigate to the **nps-agent** experiment. Every request is auto-traced — you'll see the full chain of LLM calls and MCP tool invocations.

## Rebuilding After Code Changes

Push changes to the `deploydemo` branch, then trigger a new build:

In [ ]:
!oc start-build nps-agent -n {NAMESPACE}
!oc logs -f build/nps-agent-2 -n {NAMESPACE}

## Cleanup

To delete everything and start from scratch:

In [176]:
!oc delete project {NAMESPACE}

project.project.openshift.io "nehanth" deleted
